# LLM Classification Finetuning — v0.3 Gemma-2-9b TPU v5e-8

**Approach:** `google/gemma-2-9b-it` fine-tuned with LoRA on Kaggle TPU v5e-8 (8 chips × 16 GB = 128 GB).
Pure bf16 — no quantization needed. Label-token readout at inference → 3-class probabilities.

**Strategy:**
- 5-fold stratified split; single fold first to validate pipeline
- A/B swap augmentation to fight position bias
- Swap-TTA at inference: average (A,B) + (B,A) predictions
- Accelerator: TPU v5e-8 (must be selected manually in Kaggle UI after push)

**Why TPU over T4:** Gemma-2-9b in bf16 (~18 GB) exceeds T4 16 GB limit without quantization.
TPU v5e-8 provides 128 GB total; Accelerate distributes data across all 8 chips automatically.

**Model source:** `google/gemma-2/transformers/gemma-2-9b-it/1` — attach via Kaggle UI first (license).

**Push workflow:**
1. `zsh scripts/push_notebook.sh v0.3-gemma-tpu`
2. Stop the auto-run immediately in Kaggle UI
3. Accelerator → TPU v5e-8 → Save Version → Save & Run All

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44', 'peft>=0.12', 'trl>=0.10',
    'accelerate>=0.33', 'datasets>=2.20',
], check=True)

print('Dependencies installed')

In [ ]:
# ── Cell 2: Imports & device detection ────────────────────────────────────────
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# Detect TPU vs GPU — Accelerate/SFTTrainer handles distribution automatically.
# Do NOT set CUDA_VISIBLE_DEVICES here; TPU placement is managed by XLA.
try:
    import torch_xla.core.xla_model as xm
    DEVICE_TYPE = 'tpu'
    print(f'TPU detected: {xm.xla_device()}')
except ImportError:
    DEVICE_TYPE = 'cuda' if torch.cuda.is_available() else 'cpu'
    if DEVICE_TYPE == 'cuda':
        print(f'GPU detected: {torch.cuda.get_device_name(0)}')
    else:
        print('CPU only')

print(f'Device type: {DEVICE_TYPE}')
print('Imports OK')

In [ ]:
# ── Cell 3: Path discovery & hyperparameters ──────────────────────────────────
OUTPUT = Path('/kaggle/working')

# Locate competition data
_data_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _data_candidates if (p / 'train.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if _found:
        INPUT = Path(_found[0]).parent

# Locate Gemma-2-9b model (mounted from Kaggle Models)
_gemma_configs = [
    p for p in glob.glob('/kaggle/input/**/config.json', recursive=True)
    if 'gemma' in p.lower()
]
MODEL_PATH = str(Path(_gemma_configs[0]).parent) if _gemma_configs else None

print(f'Input path : {INPUT}')
print(f'Model path : {MODEL_PATH}')

if MODEL_PATH is None:
    print('Gemma model not found. Contents of /kaggle/input/:')
    for p in sorted(Path('/kaggle/input').rglob('config.json')):
        print(f'  {p}')

assert INPUT is not None, 'Competition data not found'
assert MODEL_PATH is not None, 'Gemma model not found — attach google/gemma-2/transformers/gemma-2-9b-it/1 via Kaggle UI'

# Hyperparameters
N_FOLDS      = 5
TRAIN_FOLD   = 0      # single fold first; set to 'all' for full 5-fold run
MAX_SEQ_LEN  = 1536   # more headroom than v0.2; Gemma-2 supports up to 8192
BATCH_SIZE   = 2      # per chip; 8 chips → 16 effective
GRAD_ACCUM   = 4      # effective batch = 64
NUM_EPOCHS   = 1
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42
LABEL_TOKENS = ['A', 'B', 'tie']

print('Config OK')

In [ ]:
# ── Cell 4: Load data ─────────────────────────────────────────────────────────
train = pd.read_csv(INPUT / 'train.csv')
test  = pd.read_csv(INPUT / 'test.csv')

winner_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
train['label'] = train[winner_cols].values.argmax(axis=1).astype(int)
train['label_str'] = train['label'].map({0: 'A', 1: 'B', 2: 'tie'})

print(f'Train: {train.shape}  Test: {test.shape}')
print(train['label_str'].value_counts())

In [ ]:
# ── Cell 5: Prompt builder ────────────────────────────────────────────────────
SYSTEM = (
    'You are an expert judge evaluating AI assistant responses. '
    'Given a user prompt and two responses (A and B), output which response '
    'a human would prefer: A, B, or tie.'
)

def build_prompt(row, swap=False):
    ra = str(row['response_a'] or '')
    rb = str(row['response_b'] or '')
    if swap:
        ra, rb = rb, ra
    prompt = str(row['prompt'] or '')
    max_resp_chars = MAX_SEQ_LEN * 3
    return (
        f'PROMPT:\n{prompt}\n\n'
        f'RESPONSE A:\n{ra[:max_resp_chars]}\n\n'
        f'RESPONSE B:\n{rb[:max_resp_chars]}\n\n'
        f'Which response is better? Answer with only: A, B, or tie.'
    )

def label_for_swap(label_str, swap):
    if not swap or label_str == 'tie':
        return label_str
    return 'B' if label_str == 'A' else 'A'

print('Prompt builder ready')

In [ ]:
# ── Cell 6: Tokenizer + fold split + pre-tokenized training set ───────────────
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.model_max_length = MAX_SEQ_LEN
print(f'Tokenizer loaded from {MODEL_PATH}')

def make_chat_text(row, swap=False):
    messages = [
        {'role': 'user',  'content': build_prompt(row, swap)},
        {'role': 'model', 'content': label_for_swap(row['label_str'], swap)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train, train['label']))
tr_idx, val_idx = splits[TRAIN_FOLD]

df_tr  = train.iloc[tr_idx].reset_index(drop=True)
df_val = train.iloc[val_idx].reset_index(drop=True)

train_texts = []
for _, row in df_tr.iterrows():
    train_texts.append(make_chat_text(row, swap=False))
    train_texts.append(make_chat_text(row, swap=True))

print(f'Fold {TRAIN_FOLD}: train={len(df_tr)} ({len(train_texts)} w/ swap aug)  val={len(df_val)}')
print('Pre-tokenizing dataset (parallel) …')

# Pre-tokenize on CPU before the training cell allocates the TPU.
# SFTTrainer's internal single-threaded tokenization (~2h for 91k examples)
# triggers Kaggle's 2h TPU idle timeout — see docs/investigate/2026-06-27-v02-t4x2-ddp.md §2.
raw_ds = Dataset.from_dict({'text': train_texts})

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_ds = raw_ds.map(
    tokenize_fn,
    batched=True,
    batch_size=1000,
    num_proc=4,
    remove_columns=['text'],
    desc='Tokenizing',
)
print(f'Tokenization complete — {len(train_ds):,} examples ready')

In [ ]:
# ── Cell 7: Load Gemma-2-9b + LoRA (bf16, no quantization) ───────────────────
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# Gemma-2-9b in bf16 = ~18 GB. With TPU v5e-8 (128 GB total) this fits on a
# single chip; Accelerate distributes data across all 8 chips automatically.
# Do NOT set device_map — XLA handles placement via Accelerate.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
    attn_implementation='eager',   # sdpa/flash_attn not supported on TPU
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

assert DEVICE_TYPE == 'tpu', (
    f'Training requires TPU v5e-8 (got {DEVICE_TYPE!r}). '
    'Stop this auto-run in the Kaggle UI → Accelerator → TPU v5e-8 → Save & Run All.'
)

output_dir = OUTPUT / f'fold{TRAIN_FOLD}-adapter'

sft_config = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    bf16=True,
    fp16=False,
    gradient_checkpointing=False,   # can cause XLA recompilation issues
    dataloader_drop_last=True,      # XLA requires static shapes
    logging_steps=25,
    save_strategy='epoch',
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    seed=SEED,
    report_to='none',
    # dataset is pre-tokenized in cell-6 — skip SFTTrainer's internal tokenization
    dataset_text_field='',
    max_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(str(output_dir))
print(f'Adapter saved → {output_dir}')

In [ ]:
# ── Cell 9: Inference helper (label-token readout) ────────────────────────────
label_token_ids = [
    tokenizer.encode(t, add_special_tokens=False)[0]
    for t in LABEL_TOKENS
]
print(f'Label token ids: {dict(zip(LABEL_TOKENS, label_token_ids))}')

# Move model to device for inference (handles both TPU and GPU gracefully)
if DEVICE_TYPE == 'tpu':
    import torch_xla.core.xla_model as xm
    infer_device = xm.xla_device()
else:
    infer_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(infer_device)

def predict_proba(df, batch_size=8, swap=False):
    model.eval()
    all_probs = []
    rows = [r for _, r in df.iterrows()]
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{'role': 'user', 'content': build_prompt(row, swap=swap)}],
                tokenize=False, add_generation_prompt=True
            )
            for row in batch
        ]
        enc = tokenizer(
            texts, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN
        ).to(infer_device)
        with torch.no_grad():
            last_logits = model(**enc).logits[:, -1, :]
        probs = torch.softmax(
            last_logits[:, label_token_ids].float(), dim=-1
        ).cpu().numpy()
        all_probs.append(probs)
        if (i // batch_size) % 50 == 0:
            print(f'  {i+len(batch)}/{len(rows)}')
    return np.vstack(all_probs)

print('Inference helper ready')

In [ ]:
# ── Cell 10: OOF evaluation ───────────────────────────────────────────────────
print('OOF inference (swap-TTA) …')
probs_fwd = predict_proba(df_val, swap=False)
probs_swp = predict_proba(df_val, swap=True)
oof_probs = (probs_fwd + probs_swp[:, [1, 0, 2]]) / 2

oof_ll = log_loss(df_val['label'].values, oof_probs)
print(f'Fold {TRAIN_FOLD} OOF log loss (swap-TTA): {oof_ll:.4f}')

In [ ]:
# ── Cell 11: Test inference & submission ──────────────────────────────────────
print('Test inference (swap-TTA) …')
test_fwd = predict_proba(test, swap=False)
test_swp = predict_proba(test, swap=True)
test_probs = (test_fwd + test_swp[:, [1, 0, 2]]) / 2

sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie':     test_probs[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'submission.csv written — {len(sub):,} rows')
print(sub.head())
print(f'Prob sums: {test_probs.sum(axis=1)[:5].round(4)}')
print(f'OOF fold {TRAIN_FOLD}: {oof_ll:.4f}  (v0.1 baseline: 1.0404)')